
# C7-cnn-transfer — Session 3: Truncation and Transfer

*One class session, roughly 85 minutes. Prerequisites: Sessions 1–2
(feature maps, the feature hierarchy, ResNet-50's anatomy, stage
shapes, parameter arithmetic) and C6 (modules, `requires_grad`,
`named_parameters`).*

**This session:** ResNet-50 as a *parts supplier*.
Its pretrained body computes general-purpose visual features — Session
1's hierarchy said the early ones (edges, textures) are barely
task-specific at all — so instead of building vision models from
scratch, practitioners **truncate** the network to the depth whose
features they want, **freeze** what they keep, and bolt on a fresh
head for their own task: **transfer learning**.
This session is the engineering of that move: `nn.Sequential` slices
over `children()`, `requires_grad` loops at scale, frozen/trainable
audits, and a complete worked transfer build.

> **Scope note (course fence).**
> *Training* the fresh head — actually fitting its weights to a
> dataset — needs machinery beyond this course (loss functions,
> gradients, optimizers).
> The exam grades the **construction**: the right layers kept, the
> right flags set, the right shapes and counts — everything this
> session builds and audits.
> Where a fresh head's weights land after training is someone else's
> chapter; that the construction is *ready to train* is yours.


In [ ]:

# Cache pin (course convention, plan 009): pretrained weights live in the repo's
# gitignored reference/cache/ -- resolve it from the repo root BEFORE importing torch.
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["TORCH_HOME"] = str(_root / "reference" / "cache" / "torch")

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# float32 register (course exception, stated once per notebook): pretrained
# resnet50 is a float32 artifact, so this notebook does NOT set the float64
# default. Inputs are cast .to(torch.float32) at the model boundary; float
# comparisons state atol=1e-6 / rtol=1e-5.
SEED = 20260804

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval()
print("loaded; eval:", not model.training)



## 1. Why Reuse a Trained Network?

Two facts, both already yours:

1. **The features are general.**
   Session 1 §8: early layers detect edges and textures — patterns of
   *images*, not of ImageNet's 1000 classes.
   Even middle layers (motifs, parts) transfer across tasks far better
   than random weights would.
   (This generality claim is an empirical fact about trained vision
   networks, stated here as such — the course's evidence register for
   it is the hierarchy's statistics, not a proof.)
2. **The features are expensive.**
   25.5 million parameters (Session 2 §6) distilled from over a
   million labeled images.
   A project with 900 labeled photos cannot re-earn that from scratch
   — but it can *inherit* it.

So the recipe: keep the pretrained body as a **frozen feature
extractor**, discard the 1000-class head (it answers the *wrong
question*), and attach a fresh head sized for the new task.
Only the head would ever need fitting — and it is tiny (the $2049k$
formula from Session 2).

The three constructions this session drills:

| move | tool | section |
|---|---|---|
| **truncate** — keep a prefix of the network | `nn.Sequential(*children_list[:k])` | §2, §3 |
| **freeze** — mark kept weights as fixed | `requires_grad=False` loops + audits | §4 |
| **graft** — attach a fresh head | `nn.Linear` after pool/flatten, or replace `model.fc` | §5, §6, §7 |

### Checkpoint 1

1. Give both facts (generality, cost) in one sentence each, and name
   which session/section established each.
2. Why is the pretrained `fc` layer specifically *useless* for a new
   9-class task, when every other layer is worth keeping?
3. Under the scope note, which of these is this course's business and
   which is not: choosing the cut depth; setting `requires_grad`
   flags; picking a learning rate; verifying the head's output shape?



## 2. Truncation: `nn.Sequential` over a Slice of `children()`

Session 2's anatomy loop becomes a construction tool.
`list(model.children())` materializes the ten top-level pieces *in
forward order* (list once — the generator pitfall); a slice of that
list, splatted into `nn.Sequential`, **is** the truncated network,
sharing the very same pretrained modules (no copy is made):


In [ ]:

ch = list(model.children())
print("pieces:", [type(c).__name__ for c in ch])

trunk_l2 = nn.Sequential(*ch[:6])     # stem + layer1 + layer2
trunk_l3 = nn.Sequential(*ch[:7])     # stem + layer1 + layer2 + layer3

torch.manual_seed(SEED)
x = torch.randn(2, 3, 224, 224).to(torch.float32)

with torch.inference_mode():
    f2 = trunk_l2(x)
    f3 = trunk_l3(x)
print("through layer2:", tuple(f2.shape))
print("through layer3:", tuple(f3.shape))



`(2, 512, 28, 28)` and `(2, 1024, 14, 14)` — Session 2's shape table,
now produced by models *ending* at those depths.
The slice index is worth reading out loud: `ch[:6]` keeps positions
0–5 (`conv1, bn1, relu, maxpool, layer1, layer2`) — "through
`layer2`" means *six* pieces because the stem is four.

**Why not simply run the full model?**
Because the full model's `forward` ends in `avgpool → flatten → fc`:
its output is 1000 class scores with all spatial structure gone.
Truncation is how you get at the *feature maps* — and note the
deliberate omission: a bare `nn.Sequential(*ch)` of all ten pieces is
**not** the model (the flatten between `avgpool` and `fc` lives in
`forward`, not in any child):


In [ ]:

broken = nn.Sequential(*ch)           # all 10 children, no flatten anywhere
try:
    with torch.inference_mode():
        broken(x)
except RuntimeError as e:
    print("RuntimeError:", str(e)[:80], "...")



The `(2, 2048, 1, 1)` tensor out of `avgpool` hits `fc` still 4-D and
the matmul refuses.
Rebuilding the full pipeline as a `Sequential` needs an explicit
`nn.Flatten(1)` in the list — Section 6's build does exactly that.

### Checkpoint 2

1. Which slice produces features of shape `(B, 256, 56, 56)`, and
   what are its pieces by name?
2. `trunk_l2[4]` and `model.layer1` — same object or copies?
   Propose a one-line identity check and state the memory
   consequence of the answer.
3. Explain to a teammate in two sentences why `nn.Sequential(*ch)`
   crashes but `model(x)` works, given that both "contain the same
   layers".



## 3. Cutting Inside a Stage

Each `layerN` is itself an `nn.Sequential` of blocks, and slicing an
`nn.Sequential` yields an `nn.Sequential` — so mid-stage cuts compose
with top-level slices:


In [ ]:

trunk_mid = nn.Sequential(*ch[:5], model.layer2[:2])   # stem + layer1 + first 2 blocks of layer2
with torch.inference_mode():
    fm = trunk_mid(x)
print("layer2[:2] slice type:", type(model.layer2[:2]).__name__,
      " len:", len(model.layer2[:2]))
print("through layer2[:2]:", tuple(fm.shape))



`(2, 512, 28, 28)` — the *same shape* as the full `layer2` trunk,
because block 0 of a stage does all the reshaping (channels up, grid
down; Session 2 §4) and blocks 1+ refine in place.
A mid-stage cut therefore changes the *depth of processing*, not the
interface — which is exactly why cut points are an engineering choice:
shallower cuts keep more spatial detail and more generic features;
deeper cuts hand you more abstraction per Session 1 §8's hierarchy.

One caution: a cut *must* respect block boundaries.
Blocks are the atomic unit — `relu(F(x) + x)` needs all of `F` — and
slicing below the block level (into a block's conv/bn internals) would
sever the skip connection and compute something no ResNet computes.

### Checkpoint 3

1. Build (on paper) the trunk "stem + layer1 + layer2 + first 3
   blocks of layer3": the slice expression, and the output shape for
   a `(1, 3, 224, 224)` input.
2. Why does `layer3[1:]` *not* make a usable trunk piece on its own
   input — what would you have to feed it, and why is that shape
   never available *before* `layer3[0]` runs?
3. State the block-boundary rule and what specifically breaks
   (which operation loses which operand) if you slice into a block's
   internals.



## 4. Freezing at Scale

C6 froze parameters one `nn.Parameter(..., requires_grad=False)` at a
time — construction-time freezing.
A pretrained network arrives already built, flags up
(`requires_grad=True`, every one of them), so here freezing is a
*loop over an existing model*:


In [ ]:

n_before = sum(p.requires_grad for p in trunk_l3.parameters())

for p in trunk_l3.parameters():
    p.requires_grad = False            # freezing at scale: flip flags in a loop

n_after = sum(p.requires_grad for p in trunk_l3.parameters())
n_tensors = sum(1 for _ in trunk_l3.parameters())
print(f"trainable tensors before: {n_before}/{n_tensors}, after: {n_after}/{n_tensors}")
print("fully frozen:", all(not p.requires_grad for p in trunk_l3.parameters()))



`129/129` before — every parameter tensor of the trunk arrived
trainable-by-intent — and `0/129` after; the C6 audit idiom
(`all(not p.requires_grad ...)`) confirms in one line.
Because `trunk_l3` *shares* modules with `model`, those 129 tensors
are now frozen in `model` too — sharing cuts both ways, and audits,
not assumptions, are how you stay honest about flag state.

For *selective* freezing, filter `named_parameters()` by name prefix
— the dotted names Session 2 read are also handles:


In [ ]:

# thaw everything first (undo the blanket freeze above), then freeze stem+layer1 only
for p in model.parameters():
    p.requires_grad = True

for name, p in model.named_parameters():
    if name.startswith(("conv1", "bn1", "layer1")):
        p.requires_grad = False

frozen_scalars = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable_scalars = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"frozen scalars:    {frozen_scalars:>10,}")
print(f"trainable scalars: {trainable_scalars:>10,}")
print("split adds up:", frozen_scalars + trainable_scalars == 25_557_032)

trainable_tops = sorted({name.split(".")[0]
                         for name, p in model.named_parameters() if p.requires_grad})
print("still trainable at top level:", trainable_tops)



`225,344` frozen ($9{,}408 + 128 + 215{,}808$ — stem conv, its BN,
all of `layer1`) against `25,331,688` trainable, summing to the model
total; the name-prefix audit lists exactly the untouched tops.
The two audit currencies — **tensors** (how many parameter objects)
and **scalars** (how many numbers) — both matter on the exam; keep
`sum(1 for ...)` and `sum(p.numel() for ...)` distinct in your
fingers.

A flag-register reminder from C6, sharpened for pretrained models:
`requires_grad` records **intent**; it changes no forward value.
The freeze loop is how a construction *documents* "these weights are
inherited and fixed; only the head is meant to move" — and the audit
is how a grader (or CI) reads that intent back out.

### Checkpoint 4

1. Freeze *only* `layer4` and `fc` (on paper): the loop, the two
   audit numbers you'd print, and their values from Session 2 §6's
   table.
2. `sum(p.requires_grad for p in ...)` and
   `sum(1 for p in ... if p.requires_grad)` — same number? Why does
   the first one work at all?
3. After this section's selective freeze, `trunk_l2` (built in §2)
   shares modules with `model`.
   Which of *its* parameters are now frozen, and what does that say
   about auditing a trunk you built before a freeze ran elsewhere?



## 5. Transfer Learning: Frozen Backbone, Fresh Head

Assemble the moves.
A transfer model for a $k$-class task over an $f$-channel trunk:

1. **backbone** — a truncated, *frozen* prefix of the pretrained
   network (ends in feature maps, $f$ channels);
2. **pool + flatten** — `nn.AdaptiveAvgPool2d(1)` then
   `nn.Flatten(1)`: feature maps to a $(B, f)$ vector, any image
   size;
3. **head** — a fresh `nn.Linear(f, k)`: randomly initialized,
   *trainable by intent* (its flags stay `True`), holding
   $f \cdot k + k$ parameters.

The division of labor is the whole design: everything expensive is
inherited and frozen; everything task-specific is small and fresh.
The audit signature of a correct transfer construction — and what
exam problems actually grade:

- forward shape: input $(B, 3, H, W)$ → output $(B, k)$;
- flags: backbone all-`False`, head all-`True`;
- counts: frozen scalars = the backbone's (hand-computable from
  Session 2 §6); trainable = $f\cdot k + k$.

### Checkpoint 5

1. For a backbone through `layer2` ($f = 512$) and $k = 9$: the three
   audit values (frozen scalars, trainable scalars, output shape) —
   hand arithmetic from Session 2 §6's table.
2. Why must the pool sit *before* the head — what shape reaches the
   `Linear` without it, and which session-2 pitfall does that echo?
3. The fresh head's weights are random garbage until someone trains
   them, yet the construction is gradeable *now*.
   Name the three graded surfaces (shape/flags/counts) and say why
   none of them depends on the head's values.



## 6. Worked Exam-Style Build (constrained coding)

---

*Build `TransferNet`, an `nn.Module` for a **7-class** task on
**layer3 features**, to this contract:*

- *constructor takes the pretrained `model`; attributes exactly:
  `backbone` — `nn.Sequential` of the children through `layer3`;
  `pool` — `nn.AdaptiveAvgPool2d(1)`; `head` — `nn.Linear(1024, 7)`;*
- *the constructor freezes every backbone parameter (and only
  those);*
- *`forward(x)` = backbone → pool → flatten(1) → head;*
- *deliverables: `net7` (an instance), `out7` — its output on the
  seeded `(2, 3, 224, 224)` batch under `inference_mode`;
  `n_frozen`, `n_trainable` — scalar counts; `trainable_names` — the
  sorted names of trainable parameter tensors.*
- ***Banned (zero points): `torchvision.models.feature_extraction`;
  forward hooks; mutating the global `model`'s `fc`.***

*Predict `n_frozen` and `n_trainable` by hand before running
(Session 2 §6 + the $f\cdot k + k$ formula).*

---

*Solution.*
Hand prediction first: frozen = stem + `layer1..3`
$= 9{,}408 + 128 + 215{,}808 + 1{,}219{,}584 + 7{,}098{,}368 =
8{,}543{,}296$; trainable $= 1024 \cdot 7 + 7 = 7{,}175$.


In [ ]:

class TransferNet(nn.Module):
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = nn.Sequential(*list(pretrained.children())[:7])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(1024, 7)
        for p in self.backbone.parameters():
            p.requires_grad = False

    def forward(self, x):
        f = self.pool(self.backbone(x))
        return self.head(torch.flatten(f, 1))


for p in model.parameters():
    p.requires_grad = True             # reset §4's experiment before building

net7 = TransferNet(model)
net7.eval()

with torch.inference_mode():
    out7 = net7(x)                     # x is already float32 (the boundary cast)

n_frozen = sum(p.numel() for p in net7.parameters() if not p.requires_grad)
n_trainable = sum(p.numel() for p in net7.parameters() if p.requires_grad)
trainable_names = sorted(n for n, p in net7.named_parameters() if p.requires_grad)

print("out7:", tuple(out7.shape))
print(f"n_frozen: {n_frozen:,}   n_trainable: {n_trainable:,}")
print("trainable tensors:", trainable_names)
assert n_frozen == 8_543_296 and n_trainable == 7_175



`(2, 7)`, `8,543,296` frozen, `7,175` trainable — the hand
prediction, bit for bit — and the trainable list reads
`['head.bias', 'head.weight']`: the audit signature of a clean
transfer construction, in three printed lines.

Note what the contract *never asked for*: the values in `out7`.
They pass through a randomly initialized head — meaningful only
after training, which is beyond the fence.
Construction, flags, shapes, counts: graded; predictions: not.

### Checkpoint 6

1. Rework the build for **12 classes on layer2 features**: which two
   constructor lines change, and the new `n_frozen`/`n_trainable` by
   hand?
2. Why does the contract pin `trainable_names` and not just
   `n_trainable` — what construction bug does the *name* audit catch
   that the count could miss?
3. The ban list forbids mutating the global `model`'s `fc`.
   What goes wrong for *other* users of `model` if a build does that,
   and which section's sharing warning is this?



## 7. The Other Graft: Replacing `model.fc` In Place

When the new task wants the **full** backbone depth (everything
through `avgpool`), there is a shorter route than rebuilding:
overwrite the `fc` attribute.
Attribute assignment on a module *re-registers* the child (C6 §
submodule registration), so the ten-piece pipeline — flatten and all
— keeps working with the new final layer in place.
Do it on a **fresh copy** of the model (the sharing lesson: never
mutate the artifact other code is using):


In [ ]:

import copy

surgical = copy.deepcopy(model)
surgical.eval()

for p in surgical.parameters():
    p.requires_grad = False            # freeze everything...
surgical.fc = nn.Linear(2048, 13)      # ...then graft: fresh fc, flags True by birth

with torch.inference_mode():
    out13 = surgical(x)

n_tr = sum(p.numel() for p in surgical.parameters() if p.requires_grad)
n_fr = sum(p.numel() for p in surgical.parameters() if not p.requires_grad)
print("output:", tuple(out13.shape))
print(f"trainable: {n_tr:,}  (= 2049*13 = {2049*13:,})")
print(f"frozen:    {n_fr:,}  total: {n_tr + n_fr:,}")



`(2, 13)` out of the intact pipeline; trainable `26,637` — the
$2049k$ formula at $k = 13$ — and the model's new total
`23,534,669` = $25{,}557{,}032 - 2{,}049{,}000 + 26{,}637$: surgery
*removed* the 1000-class head's two million parameters and installed
a thirteenth-of-the-size one.

Order matters and is worth reading off the code: **freeze first, then
graft.**
The fresh `nn.Linear` is born with `requires_grad=True`, so grafting
after the freeze leaves exactly the head trainable — one loop, no
exceptions to track.
(The reverse order works too but needs a second, fiddlier loop to
re-thaw the head; Section 8 shows the bug the careless version of
that plants.)

**Slice-build or surgery?**
Surgery keeps the full depth and the existing forward (flatten
included) — minimal code when you want everything through `avgpool`.
Slice-builds choose their depth freely and make the architecture
explicit — the tool when the cut is anywhere *other* than the very
end.
Both are graded by the same audit signature.

### Checkpoint 7

1. Perform (on paper) surgery for a 4-class task: the two lines after
   the freeze loop, plus expected trainable count and model total.
2. Why is the grafted head *not* frozen by the loop that ran two
   lines earlier?
   What single reordering would (silently) freeze it, and what would
   the audit print then?
3. You need `(B, 512, 28, 28)` feature maps for a detection-style
   task.
   Slice-build or surgery, and why is the other one simply unable to
   produce this?



## 8. Common Pitfalls III

**Pitfall 1 — the freeze loop that eats the head.**
Freeze *the whole assembled model* after attaching the head, and the
head freezes with it — the construction is silently untrainable:


In [ ]:

oops = copy.deepcopy(model)
oops.fc = nn.Linear(2048, 13)          # graft first...
for p in oops.parameters():            # ...freeze second: WRONG ORDER
    p.requires_grad = False

n_tr_oops = sum(p.numel() for p in oops.parameters() if p.requires_grad)
print("trainable parameters in the botched build:", n_tr_oops)
print("the audit catches it: expected 26,637, got", n_tr_oops)
del oops



`0` trainable.
No error is raised anywhere — the forward pass is perfectly happy —
which is precisely why the **audit is part of the construction**:
a transfer build is not done until `n_trainable` prints the head's
count and nothing else's.

**Pitfall 2 — pool but no flatten.**
`nn.AdaptiveAvgPool2d(1)` outputs $(B, f, 1, 1)$, not $(B, f)$;
feeding that to `nn.Linear` is Session 2's flatten crash reborn in
your own build:


In [ ]:

no_flatten = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Linear(512, 9))
try:
    with torch.inference_mode():
        no_flatten(torch.zeros(2, 512, 28, 28))
except RuntimeError as e:
    print("RuntimeError:", str(e)[:75], "...")



The fix is one `nn.Flatten(1)` between them — in a `Sequential`
build, as a module in the list; in a custom `forward`, as
`torch.flatten(f, 1)` (Section 6 used the latter).

**Pitfall 3 — a float64 head grafted onto a float32 body.**
This unit's notebooks leave the float32 default in place, so fresh
heads are born float32 and match.
But in a notebook that had run the course's
`torch.set_default_dtype(torch.float64)` header, the *same graft
line* births a float64 head — and the forward crashes where body
meets head:


In [ ]:

mixed = nn.Linear(2048, 5).to(torch.float64)   # what a float64-default notebook builds
try:
    with torch.inference_mode():
        mixed(torch.zeros(2, 2048, dtype=torch.float32))   # float32 features arrive
except RuntimeError as e:
    print("RuntimeError:", str(e)[:75], "...")
print("register reminder: in resnet notebooks the artifact is float32 -- head,")
print("inputs, and comparisons all live there (atol=1e-6/rtol=1e-5 when comparing).")



Same dtype-boundary crash as Session 2 §7, now *inside* your own
construction: the head must live in the artifact's dtype.
The unit's rule of thumb — **in any notebook that touches the
pretrained model, the float32 register governs end to end.**

**Pitfall 4 — trusting flags to do inference_mode's job (or vice
versa).**
`requires_grad=False` documents intent; it does **not** stop autograd
bookkeeping on the *head* (whose flags are `True`), and
`inference_mode()` stops bookkeeping but flips no flags.
A construction needs both: flags for the audit (what is *meant* to
move), `inference_mode` for every forward (nothing is moving *now*):


In [ ]:

flags = sorted({p.requires_grad for p in net7.parameters()})
print("net7 flag values present:", flags, "(mixed by design)")
with torch.inference_mode():
    y = net7(x)
print("and the audit still reads intent:",
      sum(p.numel() for p in net7.parameters() if p.requires_grad), "trainable")



`[False, True]` — a healthy transfer model is *supposed* to be
mixed-flag; the construction's honesty lives in the audit plus the
`inference_mode` habit, not in an all-frozen model.

### Checkpoint 8

1. A teammate's transfer build prints `n_trainable = 0` yet its
   forward works and its outputs "look fine".
   Which pitfall, what reorder fixes it, and why did nothing crash?
2. Predict the printed error type and cause:
   `nn.Sequential(*list(model.children())[:9], nn.Linear(2048, 6))`
   on a `(2, 3, 224, 224)` input.
3. State pitfall 4's division of labor in one sentence per tool
   (flags vs `inference_mode`), and say which audit each one feeds.



## Exam Connections

How this unit's material shows up in Round 1 (paraphrased from the
`reference/analysis.md` topic table — no real test text here):

- The **PyTorch engineering cluster** (7 sub-parts, 50 points in
  r1-2026) includes **ResNet surgery and transfer learning**
  end-to-end: truncating a pretrained torchvision ResNet at a stated
  cut point, freezing at scale, attaching a fresh head for a given
  class count, and auditing shapes/flags/counts — Sessions 2–3's
  register exactly, including the constructions-not-training fence.
- The **CNN representations cluster** (10 points) asks for the
  feature-hierarchy reasoning of Session 1 §8 — ordering depth
  against abstraction — in a form that grades the *register*
  (early = local/high-frequency, late = semantic/sparse), not any
  specific figure.
- **Parameter arithmetic on real blocks** (Session 2 §5's
  count-without-`numel` register, bottleneck structure, the $2049k$
  head formula) is among the paper's most bankable point clusters —
  if the layer/conv/BN atoms are automatic.
- The constrained-coding texture matches Section 6's worked build:
  exact attribute names, exact slice contracts, API bans with
  zero-point clauses, and audits as deliverables.

## Going Deeper

Optional forward pointers along the course map — nothing here is
needed for this unit's practice:

- **`C10-competition-craft`**: the transfer constructions built here
  are the standard opening move of applied-vision competition
  pipelines; C10 turns construction into *strategy* — when to cut
  deep versus shallow, how to budget parameters against data size,
  and how such choices are graded in the notebook task's register.
- The truncation idiom (prefix of `children()`, features instead of
  logits) is also how practitioners extract *embeddings* from any
  pretrained network — C8 studies embeddings in their own right, on
  the word side, where they arrive pre-truncated.



## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. Generality: early/middle layers detect edges, textures, and motifs
   that belong to images in general, not to ImageNet's classes
   (Session 1 §8).
   Cost: the 25.5M-parameter body distills a scale of data no small
   project has (Session 2 §6).
2. `fc` maps 2048 features to *ImageNet's 1000 specific classes* —
   its weights answer a question the new task never asks; every
   earlier layer computes reusable features instead.
3. Course business: choosing the cut, setting flags, verifying
   shapes.
   Not: picking a learning rate (training machinery, beyond the
   fence).

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. `nn.Sequential(*ch[:5])` — `conv1, bn1, relu, maxpool, layer1` —
   features `(B, 256, 56, 56)`.
2. Same object: `trunk_l2[4] is model.layer1` prints `True` —
   `nn.Sequential(*slice)` stores references, not copies.
   Consequence: no extra memory for trunks — but any flag/weight
   change through one name is visible through all.
3. `model(x)` runs `model.forward`, which calls
   `torch.flatten(x, 1)` between `avgpool` and `fc`;
   `nn.Sequential` only chains children and no child performs the
   flatten, so `fc` receives a 4-D tensor and the matmul fails.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. `nn.Sequential(*ch[:6], model.layer3[:3])` (note: through
   `layer2` is `ch[:6]`); output `(1, 1024, 14, 14)` — `layer3[0]`
   already reshapes to the stage's interface.
2. `layer3[1:]` starts at an *interior* block, which expects the
   stage's own interface `(B, 1024, 14, 14)` — a shape that only
   exists *after* `layer3[0]` has run (before it, the tensor is
   `(B, 512, 28, 28)`).
3. Blocks are atomic: each computes `relu(F(x) + x)`.
   Slicing into a block's internals keeps `F`'s convs but loses the
   skip addition's second operand `x` (the block input), so the sum —
   the residual structure itself — is gone.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. ```python
   for name, p in model.named_parameters():
       if name.startswith(("layer4", "fc")):
           p.requires_grad = False
   ```
   Frozen scalars $14{,}964{,}736 + 2{,}049{,}000 = 17{,}013{,}736$;
   trainable $25{,}557{,}032 - 17{,}013{,}736 = 8{,}543{,}296$.
2. Same number — `True` counts as 1 and `False` as 0 in a Python
   sum, so summing the flags *is* counting the `True`s.
3. All of `trunk_l2`'s stem and `layer1` parameters are frozen (they
   are the same objects the loop touched); its `layer2` parameters
   are not.
   Moral: audit a trunk *at use time* — its flags reflect every loop
   that ever ran on shared modules, not the state at build time.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Frozen $= 9{,}408 + 128 + 215{,}808 + 1{,}219{,}584 =
   1{,}444{,}928$; trainable $= 512 \cdot 9 + 9 = 4{,}617$; output
   `(B, 9)`.
2. Without the pool, `layer2` features arrive at the `Linear` as
   `(B, 512, 28, 28)` — 4-D against a 2-D weight: the flatten crash
   of Session 2 §3 / this session §2.
3. Shape is fixed by the architecture (layer sizes), flags by the
   freeze loop, counts by tensor shapes — all three are properties of
   the *construction*; the head's current values enter none of them.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. `self.backbone = nn.Sequential(*list(pretrained.children())[:6])`
   and `self.head = nn.Linear(512, 12)`.
   Frozen $= 1{,}444{,}928$ (Checkpoint 5's backbone); trainable
   $= 512 \cdot 12 + 12 = 6{,}156$.
2. A build that accidentally leaves (say) `layer3`'s BN tensors
   trainable *and* under-sizes the head could still hit some target
   count; the name audit demands the trainable set be exactly
   `{head.bias, head.weight}` — wrong *membership* cannot hide in a
   right-looking total.
3. Every other trunk/build sharing `model` suddenly finds a 13-class
   `fc` where 1000 classes stood — §2's sharing warning (slices hold
   references); mutate copies, not the shared artifact.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. `surgical.fc = nn.Linear(2048, 4)` after the freeze loop (on a
   fresh deepcopy); trainable $2049 \cdot 4 = 8{,}196$; total
   $25{,}557{,}032 - 2{,}049{,}000 + 8{,}196 = 23{,}516{,}228$.
2. The loop ran *before* the graft, so it never saw the new layer;
   the fresh `nn.Linear`'s flags are `True` by birth.
   Freezing after grafting flips the head too — the audit would
   print `0` trainable (Section 8, Pitfall 1).
3. Slice-build: surgery keeps the full ten-piece pipeline, whose
   output is `(B, 1000)`-shaped scores (or `(B, k)` after a head
   swap) — spatial maps are gone by `avgpool`; only a truncation
   that *stops* at `layer2` exposes `(B, 512, 28, 28)`.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Pitfall 1 (freeze after graft).
   Reorder: freeze first, graft second (or re-thaw the head).
   Nothing crashes because flags change no forward arithmetic —
   untrainability is invisible to a forward pass; only the audit sees
   it.
2. `RuntimeError` at the `Linear`: `ch[:9]` ends with `avgpool`, so
   the fresh linear receives `(2, 2048, 1, 1)` — 4-D input, the
   missing-flatten crash (the build needs `nn.Flatten(1)` inserted
   before the head).
3. Flags record which parameters are *meant* to be adjustable —
   feeding the frozen/trainable audit; `inference_mode` guarantees
   the current forward does no autograd bookkeeping — feeding no
   audit, just correctness of the run register.

</details>
